# Benchmarking NoteBook 

### First we ghave to start by loading our data 


#### necessary file orientation 

In [1]:
import sys
import os
from pathlib import Path

# Add the parent directory to Python path
notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent
sys.path.append(str(parent_dir))

# Now you can import modules from parent directory
try:
    from MSEB_based.data_loading import PCGEmbeddingDataset
    print("Successfully imported from parent directory")
except ImportError as e:
    print(f"Import failed: {e}")

Successfully imported from parent directory


In [2]:
dataset = PCGEmbeddingDataset(include_unknown=True)

X = dataset.embeddings  # (n_samples, 256)
y = dataset.labels      # (n_samples,) with values 0, 1, -1
metadata = dataset.metadata

Dataset initialized with 68104 samples

Class distribution:
  Unknown: 116 (0.2%)
  Normal: 15967 (23.4%)
  Abnormal: 52021 (76.4%)


## quick demo evaluation for TS2vec embeddings 

In [3]:
try:
    from MSEB_based.classification_evaluator import PCGEmbeddingEvaluator
    print("Successfully imported from parent directory")
except ImportError as e:
    print(f"Import failed: {e}")




evaluator = PCGEmbeddingEvaluator(
        strategy='linear_probe',
        random_state=42,
        test_size=0.2
    )
print("\n3. Running evaluation...")
results = evaluator.evaluate(
    X=X,
    y=y,
    metadata=metadata
)
    
#  Print results
print("\n4. Results:")
print(f"   Accuracy: {results['accuracy']:.3f}")
print(f"   Macro F1: {results['macro_f1']:.3f}")
print(f"   Binary AUROC: {results['auroc_binary']:.3f}")
print(f"\n   Per-class F1:")
for class_name, f1 in results['per_class_f1'].items():
    print(f"     {class_name}: {f1:.3f}")

#  Run cross-validation
print("\n5. Running 5-fold cross-validation...")
cv_results = evaluator.cross_validate(
    X=X,
    y=y,
    n_splits=5
)

print(f"\n   CV Accuracy: {cv_results['accuracy_mean']:.3f} ± {cv_results['accuracy_std']:.3f}")
print(f"   CV Macro F1: {cv_results['macro_f1_mean']:.3f} ± {cv_results['macro_f1_std']:.3f}")



Successfully imported from parent directory

3. Running evaluation...

4. Results:
   Accuracy: 0.766
   Macro F1: 0.435
   Binary AUROC: 0.336

   Per-class F1:
     normal: 0.003
     abnormal: 0.867

5. Running 5-fold cross-validation...

   CV Accuracy: 0.766 ± 0.000
   CV Macro F1: 0.435 ± 0.001


### Really poor performance 
### the reason might be the class imbalance (we'll try balancing)

In [6]:
import importlib

print("\n6. Balanced loader (oversamples minority class):")
balanced_loader = dataset.get_balanced_loader(batch_size=32)
print(f"   Batches per epoch: {len(balanced_loader)}")




def reload_module(module_name):
    """Reload a Python module by name."""
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"✅ Reloaded: {module_name}")
    else:
        print(f"❌ Module {module_name} not found")

# Usage - reload your custom module
reload_module('MSEB_based.data_loading')
reload_module('MSEB_based.classification_evaluator')


6. Balanced loader (oversamples minority class):
Balanced loader: 15967 samples per class
  Normal: 15967 available → sampling 15967
  Abnormal: 52021 available → sampling 15967
   Batches per epoch: 998
✅ Reloaded: MSEB_based.data_loading
✅ Reloaded: MSEB_based.classification_evaluator


In [7]:
from MSEB_based.data_loading import extract_from_dataloader
X_balanced, y_balanced, metadata_balanced = extract_from_dataloader(balanced_loader)

print("\n3. Running evaluation...")
results = evaluator.evaluate(
    X=X_balanced,
    y=y_balanced,
    metadata=metadata_balanced
)
    
#  Print results
print("\n4. Results:")
print(f"   Accuracy: {results['accuracy']:.3f}")
print(f"   Macro F1: {results['macro_f1']:.3f}")
print(f"   Binary AUROC: {results['auroc_binary']:.3f}")
print(f"\n   Per-class F1:")
for class_name, f1 in results['per_class_f1'].items():
    print(f"     {class_name}: {f1:.3f}")

#  Run cross-validation
print("\n5. Running 5-fold cross-validation...")
cv_results = evaluator.cross_validate(
    X=X,
    y=y,
    n_splits=5
)

print(f"\n   CV Accuracy: {cv_results['accuracy_mean']:.3f} ± {cv_results['accuracy_std']:.3f}")
print(f"   CV Macro F1: {cv_results['macro_f1_mean']:.3f} ± {cv_results['macro_f1_std']:.3f}")


  Processed batch 10/998
  Processed batch 20/998
  Processed batch 30/998
  Processed batch 40/998
  Processed batch 50/998
  Processed batch 60/998
  Processed batch 70/998
  Processed batch 80/998
  Processed batch 90/998
  Processed batch 100/998
  Processed batch 110/998
  Processed batch 120/998
  Processed batch 130/998
  Processed batch 140/998
  Processed batch 150/998
  Processed batch 160/998
  Processed batch 170/998
  Processed batch 180/998
  Processed batch 190/998
  Processed batch 200/998
  Processed batch 210/998
  Processed batch 220/998
  Processed batch 230/998
  Processed batch 240/998
  Processed batch 250/998
  Processed batch 260/998
  Processed batch 270/998
  Processed batch 280/998
  Processed batch 290/998
  Processed batch 300/998
  Processed batch 310/998
  Processed batch 320/998
  Processed batch 330/998
  Processed batch 340/998
  Processed batch 350/998
  Processed batch 360/998
  Processed batch 370/998
  Processed batch 380/998
  Processed batch 390